# PANdeMaiz Quake — Post-análisis de registros acelerométricos

Este notebook permite explorar visualmente los archivos `.bin` grabados por los nodos ESP32 **sin** necesidad del backend ni de Firebase. Es una herramienta de análisis offline que:

1. **Lee** archivos `.bin` de dos carpetas:
   - `data/aceleraciones/` — grabaciones continuas (ruido ambiental, calibración, etc.)
   - `data/eventos/` — archivos capturados tras un trigger STA/LTA en el nodo

2. **Convierte** cada archivo a:
   - MiniSEED (`.mseed`) — formato estándar para intercambio de datos sísmicos
   - SGC-ANC (`.anc`) — formato compatible con el dataset builder del proyecto

3. **Grafica** para cada archivo:
   - Serie de tiempo en **g** con eje relativo (desde 0 s)
   - Serie de tiempo en **g** con fecha/hora real del archivo
   - Serie de tiempo en **cm/s²** con eje relativo
   - Serie de tiempo en **cm/s²** con fecha/hora real
   - Espectrogramas (3 componentes: EW, NS, VER)
   - Detección STA/LTA con zonas de evento resaltadas

---

### Formato de nombre de archivo esperado

Para que el eje de fecha/hora funcione, el nombre del archivo debe contener `YYYYMMDD_HHMMSS`:

```
accel_20260515_143000.bin
evento_20260515_143512.bin
PAN01_20260515_143000.bin
```

Si el nombre no sigue este patrón, las gráficas con fecha/hora real usarán eje relativo como fallback.

### Mapeo de ejes (heredado del firmware)

| Canal firmware | Canal FDSN (MiniSEED) | Canal SGC (.anc) |
|---|---|---|
| ax | HNE (East-West) | EW (col 0) |
| ay | HNN (North-South) | NS (col 2) |
| az | HNZ (Vertical) | VER (col 1) |

## §2 — Importaciones

In [ ]:
import io
import re
import struct
import warnings
import pathlib
from datetime import datetime, timedelta, timezone

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
from scipy.signal import spectrogram as scipy_spectrogram

from obspy import Stream, Trace, UTCDateTime
from obspy.signal.trigger import classic_sta_lta, trigger_onset

print("Librerías cargadas correctamente.")

## §3 — Configuración

> **Esta es la única celda que debes editar** antes de ejecutar el notebook.
> Ajusta las rutas y el ID de estación según tu setup.

In [ ]:
# ── Rutas de entrada ─────────────────────────────────────────────────────────
DIR_ACEL    = pathlib.Path("data/aceleraciones")   # .bin de grabación continua
DIR_EVENTOS = pathlib.Path("data/eventos")          # .bin de eventos detectados

# ── Rutas de salida ───────────────────────────────────────────────────────────
DIR_OUT     = pathlib.Path("output")
DIR_MSEED   = DIR_OUT / "mseed"
DIR_ANC     = DIR_OUT / "anc"
DIR_PLOTS   = DIR_OUT / "plots"

# Crear carpetas si no existen
for d in [DIR_MSEED, DIR_ANC, DIR_PLOTS]:
    d.mkdir(parents=True, exist_ok=True)

# ── Parámetros del sensor ─────────────────────────────────────────────────────
STATION_ID = "PAN01"   # ID de la estación (máx. 5 caracteres para MiniSEED)
FS         = 200       # Hz — frecuencia de muestreo (debe coincidir con config.h)

# ── STA/LTA (idénticos a firmware y dataset builder) ─────────────────────────
STA_SAMPLES = 100      # 0.5 s a 200 Hz
LTA_SAMPLES = 2000     # 10 s a 200 Hz
TRIGGER_ON  = 2.5
TRIGGER_OFF = 1.5

# ── Espectrograma ─────────────────────────────────────────────────────────────
NPERSEG  = 128
NOVERLAP = 64

# ── Constantes del sensor (no modificar) ──────────────────────────────────────
MAGIC    = 0xDA7A1345   # número mágico del formato .bin
SCALE_G  = 0.0039       # g/LSB — ADXL345 ±2g full-resolution
G_TO_CMS2 = 980.665     # factor de conversión g → cm/s²

print("Configuración cargada.")
print(f"  Aceleraciones : {DIR_ACEL}")
print(f"  Eventos       : {DIR_EVENTOS}")
print(f"  Salida        : {DIR_OUT}")

## §4 — Parser del formato `.bin`

Reimplementación autónoma de `parse_bin()` de `backend/app/converter.py`.

### Estructura del archivo `.bin`

| Campo | Tipo | Bytes | Descripción |
|---|---|---|---|
| MAGIC | uint32 LE | 4 | `0xDA7A1345` — identifica el formato |
| VERSION | uint16 LE | 2 | 2 = sin GPS, 3 = con lat/lon |
| SAMPLE_RATE | uint16 LE | 2 | Hz (normalmente 200) |
| lat | float32 LE | 4 | Solo v3 |
| lon | float32 LE | 4 | Solo v3 |
| **Por muestra** | — | **10** | — |
| timestamp_ms | uint32 LE | 4 | ms desde inicio |
| ax | int16 LE | 2 | EW — 1 LSB = 3.9 mg |
| ay | int16 LE | 2 | NS — 1 LSB = 3.9 mg |
| az | int16 LE | 2 | VER — 1 LSB = 3.9 mg |

In [ ]:
_HEADER_V2   = 8    # MAGIC(4) + VERSION(2) + SAMPLE_RATE(2)
_HEADER_V3   = 16   # V2 + lat(4) + lon(4)
_SAMPLE_SIZE = 10   # timestamp_ms(4) + ax(2) + ay(2) + az(2)


def parse_bin(path):
    """
    Lee un archivo .bin del ESP32.

    Retorna
    -------
    header : dict  {version, sample_rate, lat, lon}
    data   : ndarray (N, 4) float32  [t_s, ax_g, ay_g, az_g]
    """
    raw = pathlib.Path(path).read_bytes()

    if len(raw) < _HEADER_V2:
        raise ValueError(f"{path}: archivo demasiado corto ({len(raw)} bytes)")

    magic, version, sr = struct.unpack_from("<IHH", raw, 0)
    if magic != MAGIC:
        raise ValueError(f"{path}: magic inválido 0x{magic:08X} (esperado 0x{MAGIC:08X})")

    if version >= 3 and len(raw) >= _HEADER_V3:
        lat, lon = struct.unpack_from("<ff", raw, 8)
        hdr_size = _HEADER_V3
    else:
        lat, lon = 0.0, 0.0
        hdr_size = _HEADER_V2

    header = {"version": version, "sample_rate": int(sr), "lat": lat, "lon": lon}

    n_samples = (len(raw) - hdr_size) // _SAMPLE_SIZE
    samples = []
    pos = hdr_size
    for _ in range(n_samples):
        ts_ms, ax, ay, az = struct.unpack_from("<Ihhh", raw, pos)
        samples.append([ts_ms / 1000.0, ax * SCALE_G, ay * SCALE_G, az * SCALE_G])
        pos += _SAMPLE_SIZE

    if not samples:
        raise ValueError(f"{path}: no contiene muestras")

    return header, np.array(samples, dtype=np.float32)


print("parse_bin() definida.")

## §5 — Conversión a MiniSEED

Genera un `obspy.Stream` con tres trazas (HNE, HNN, HNZ) listo para guardar en disco.
Extrae el `starttime` del nombre del archivo si tiene el patrón `YYYYMMDD_HHMMSS`.

In [ ]:
def _starttime_from_stem(stem):
    """Extrae UTCDateTime del nombre de archivo. Retorna UTCDateTime(0) si no hay fecha."""
    m = re.search(r"(\d{8})_(\d{6})", stem)
    if m:
        try:
            dt = datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S")
            return UTCDateTime(dt.replace(tzinfo=timezone.utc))
        except ValueError:
            pass
    return UTCDateTime(0)


def to_mseed(header, data, station_id, stem):
    """
    Convierte datos parseados a obspy.Stream (MiniSEED).

    Canales FDSN:
      HNE = ax (East-West)
      HNN = ay (North-South)
      HNZ = az (Vertical)
    """
    t0  = _starttime_from_stem(stem)
    sta = station_id[:5].upper()
    sr  = header["sample_rate"]

    st = Stream()
    for comp, col in [("E", 1), ("N", 2), ("Z", 3)]:
        tr = Trace(data=data[:, col].astype(np.float32))
        tr.stats.network      = "PAN"
        tr.stats.station      = sta
        tr.stats.channel      = f"HN{comp}"
        tr.stats.sampling_rate = sr
        tr.stats.starttime    = t0
        st.append(tr)
    return st


print("to_mseed() definida.")

## §6 — Conversión a formato `.anc` (SGC)

Genera un texto con exactamente **20 líneas de cabecera** compatibles con `parse_anc()` del dataset builder.
Datos en cm/s², columnas en orden `EW  VER  NS` (convención SGC).

In [ ]:
def to_anc(header, data, station_id, stem):
    """
    Convierte datos parseados al formato .anc del SGC/PANdeMaiz.

    Mapeo de ejes (idéntico a converter.py del backend):
      ax → EW  (columna 0)
      az → VER (columna 1)
      ay → NS  (columna 2)
    Unidades: cm/s²
    """
    fecha = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    dt_intervalo = 1.0 / header["sample_rate"]

    lines = [
        "PANDEMAIZ QUAKE — REGISTRO ACELEROMETRICO",
        f"ESTACION: {station_id}",
        f"CODIGO DE LA ESTACION: {station_id}",
        "RED: PANdeMaiz / Universidad de Antioquia — GICM",
        f"LATITUD DE LA ESTACION: {header['lat']:.6f}",
        f"LONGITUD DE LA ESTACION: {header['lon']:.6f}",
        "LATITUD DEL EVENTO: 0.000000",
        "LONGITUD DEL EVENTO: 0.000000",
        "PROFUNDIDAD (km): 0",
        "M=?",
        "DISTANCIA EPICENTRAL: ? km",
        f"INTERVALO DE MUESTREO (s): {dt_intervalo:.6f}",
        f"NUMERO DE DATOS: {len(data)}",
        "INSTRUMENTO: ESP32 + ADXL345  +-2g  3.9 mg/LSB  BP 0.1-20 Hz",
        "UNIDADES: cm/s^2  (EW  VER  NS)",
        f"VERSION BINARIO: {header['version']}",
        "PROCESAMIENTO: Filtro Butterworth 2do orden HP 0.1 Hz LP 20 Hz",
        "GENERADO POR: PANdeMaiz Quake Post_Analysis",
        f"ARCHIVO ORIGEN: {stem}.bin",
        f"FECHA CONVERSION: {fecha}",
    ]
    assert len(lines) == 20, f"La cabecera debe tener 20 líneas, tiene {len(lines)}"

    rows = []
    for row in data:
        ew  = float(row[1]) * G_TO_CMS2   # ax → EW
        ver = float(row[3]) * G_TO_CMS2   # az → VER
        ns  = float(row[2]) * G_TO_CMS2   # ay → NS
        rows.append(f"{ew:12.4f}  {ver:12.4f}  {ns:12.4f}")

    return "\n".join(lines) + "\n" + "\n".join(rows) + "\n"


print("to_anc() definida.")

## §7 — Descubrimiento de archivos `.bin`

In [ ]:
def discover_bins(folder):
    """
    Lista todos los .bin en una carpeta, imprime una tabla resumen
    y retorna la lista ordenada de Path.
    """
    folder = pathlib.Path(folder)
    files  = sorted(folder.glob("*.bin"))

    if not files:
        print(f"[AVISO] No se encontraron archivos .bin en: {folder}")
        return []

    print(f"\n{'Archivo':<45} {'Tamaño (KB)':>12} {'Muestras est.':>14}")
    print("-" * 75)
    for f in files:
        kb       = f.stat().st_size / 1024
        # Tamaño del header estimado: 16 bytes (v3); muestras = (bytes - 16) / 10
        n_est    = max(0, (f.stat().st_size - 16) // 10)
        print(f"{f.name:<45} {kb:>12.1f} {n_est:>14,}")
    print("-" * 75)
    print(f"Total: {len(files)} archivo(s)")
    return files


print("discover_bins() definida.")

## §8 — Conversión en lote (`.bin` → MiniSEED + `.anc`)

Procesa todas las carpetas y guarda los archivos convertidos en `output/mseed/` y `output/anc/`.

In [ ]:
print("=" * 60)
print("CARPETA: aceleraciones")
print("=" * 60)
bins_acel = discover_bins(DIR_ACEL)

print()
print("=" * 60)
print("CARPETA: eventos")
print("=" * 60)
bins_eventos = discover_bins(DIR_EVENTOS)

all_bins = [(p, "acel") for p in bins_acel] + [(p, "evento") for p in bins_eventos]

conversion_log = []

for path, tag in all_bins:
    stem = path.stem
    try:
        header, data = parse_bin(path)

        # — MiniSEED
        st   = to_mseed(header, data, STATION_ID, stem)
        out_mseed = DIR_MSEED / f"{stem}.mseed"
        buf  = io.BytesIO()
        st.write(buf, format="MSEED")
        out_mseed.write_bytes(buf.getvalue())

        # — ANC
        anc_text  = to_anc(header, data, STATION_ID, stem)
        out_anc   = DIR_ANC / f"{stem}.anc"
        out_anc.write_text(anc_text, encoding="utf-8")

        conversion_log.append({
            "archivo": path.name, "tipo": tag,
            "muestras": len(data),
            "dur_s": round(len(data) / header["sample_rate"], 1),
            "ver": header["version"],
            "lat": round(header["lat"], 4),
            "lon": round(header["lon"], 4),
            "mseed": out_mseed.name,
            "anc": out_anc.name,
            "ok": True,
        })
        print(f"[OK] {path.name}  →  {out_mseed.name}  +  {out_anc.name}")

    except Exception as e:
        conversion_log.append({"archivo": path.name, "tipo": tag, "ok": False, "error": str(e)})
        print(f"[ERROR] {path.name}: {e}")

print(f"\nConvertidos: {sum(1 for r in conversion_log if r['ok'])} / {len(conversion_log)} archivos.")

## §9 — Funciones auxiliares de tiempo

Cada función de graficación acepta `use_datetime=True/False` para alternar entre:
- **Eje relativo** (desde 0 s) — siempre disponible
- **Eje fecha/hora real** (extraída del nombre del archivo) — requiere patrón `YYYYMMDD_HHMMSS`

In [ ]:
def get_starttime(stem):
    """
    Extrae datetime del nombre del archivo buscando 'YYYYMMDD_HHMMSS'.
    Retorna datetime (UTC) o None si no se encuentra el patrón.
    """
    m = re.search(r"(\d{8})_(\d{6})", stem)
    if m:
        try:
            return datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S").replace(tzinfo=timezone.utc)
        except ValueError:
            pass
    return None


def make_relative_axis(data):
    """
    Retorna el eje de tiempo relativo en segundos (siempre empieza en 0.0).
    data: ndarray (N,4) donde la columna 0 es t_s.
    """
    t = data[:, 0].astype(float)
    return t - t[0]


def make_datetime_axis(data, starttime):
    """
    Construye una lista de datetime reales: starttime + t_relativo.
    Si starttime es None, emite advertencia y retorna el eje relativo.
    """
    t_rel = make_relative_axis(data)
    if starttime is None:
        warnings.warn(
            "No se encontró fecha en el nombre del archivo. "
            "Usando eje de tiempo relativo como fallback.",
            stacklevel=2,
        )
        return t_rel, False  # (eje, is_datetime)
    times = [starttime + timedelta(seconds=float(t)) for t in t_rel]
    return times, True  # (eje, is_datetime)


def _apply_xaxis(ax, xdata, use_datetime, xlabel_rel="Tiempo relativo (s)"):
    """
    Aplica formato de eje X a un Axes de matplotlib:
    - use_datetime=True  → DateFormatter HH:MM:SS, etiquetas rotadas
    - use_datetime=False → eje numérico en segundos
    """
    if use_datetime and isinstance(xdata[0], datetime):
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        plt.setp(ax.get_xticklabels(), rotation=20, ha="right", fontsize=8)
        ax.set_xlabel("Hora (HH:MM:SS UTC)", fontsize=9)
    else:
        ax.set_xlabel(xlabel_rel, fontsize=9)


print("Funciones auxiliares de tiempo definidas.")

## §10 — Funciones de graficación

Todas aceptan `use_datetime` y `starttime` para controlar el eje X.
Guardan automáticamente la figura en `output/plots/` si se provee `save_path`.

In [ ]:
# ── Colores por componente ────────────────────────────────────────────────────
_COLORS = {"EW": "#2196F3", "NS": "#4CAF50", "VER": "#F44336"}
_LABELS = ["EW  (ax)", "NS  (ay)", "VER (az)"]
_COLS   = [1, 2, 3]  # índices en data


def _save(fig, save_path):
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")


# ──────────────────────────────────────────────────────────────────────────────
def plot_timeseries(header, data, title, use_datetime, starttime, save_path=None):
    """
    Serie de tiempo en g — 3 componentes (EW, NS, VER).
    """
    if use_datetime:
        xdata, is_dt = make_datetime_axis(data, starttime)
    else:
        xdata, is_dt = make_relative_axis(data), False

    fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
    fig.suptitle(title, fontsize=12, fontweight="bold")

    for i, (ax, label, col, color) in enumerate(zip(axes, _LABELS, _COLS, _COLORS.values())):
        sig = data[:, col]
        ax.plot(xdata, sig, color=color, linewidth=0.6, alpha=0.85)
        ax.axhline(0, color="k", linewidth=0.5, linestyle="--", alpha=0.4)
        pga = np.max(np.abs(sig))
        ax.set_ylabel(f"{label}\n[g]", fontsize=9)
        ax.text(0.01, 0.97, f"PGA = {pga:.4f} g", transform=ax.transAxes,
                fontsize=8, va="top", color=color, fontweight="bold")
        ax.grid(True, alpha=0.25)
        if i == 2:
            _apply_xaxis(ax, xdata, is_dt)

    plt.tight_layout()
    _save(fig, save_path)
    plt.show()


# ──────────────────────────────────────────────────────────────────────────────
def plot_timeseries_cms2(header, data, title, use_datetime, starttime, save_path=None):
    """
    Serie de tiempo en cm/s² — 3 componentes (EW, NS, VER).
    Útil para comparar directamente con registros .anc del SGC.
    """
    if use_datetime:
        xdata, is_dt = make_datetime_axis(data, starttime)
    else:
        xdata, is_dt = make_relative_axis(data), False

    fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
    fig.suptitle(title + "  [cm/s²]", fontsize=12, fontweight="bold")

    for i, (ax, label, col, color) in enumerate(zip(axes, _LABELS, _COLS, _COLORS.values())):
        sig = data[:, col] * G_TO_CMS2
        ax.plot(xdata, sig, color=color, linewidth=0.6, alpha=0.85)
        ax.axhline(0, color="k", linewidth=0.5, linestyle="--", alpha=0.4)
        pga = np.max(np.abs(sig))
        ax.set_ylabel(f"{label}\n[cm/s²]", fontsize=9)
        ax.text(0.01, 0.97, f"PGA = {pga:.2f} cm/s²", transform=ax.transAxes,
                fontsize=8, va="top", color=color, fontweight="bold")
        ax.grid(True, alpha=0.25)
        if i == 2:
            _apply_xaxis(ax, xdata, is_dt)

    plt.tight_layout()
    _save(fig, save_path)
    plt.show()


# ──────────────────────────────────────────────────────────────────────────────
def plot_spectrogram(header, data, title, use_datetime, starttime, save_path=None):
    """
    Espectrograma STFT (Welch sliding) para las 3 componentes.
    Escala: dB re g²/Hz  (10·log10(Sxx + 1e-12)).
    """
    sr = header["sample_rate"]
    t0 = starttime if (use_datetime and starttime is not None) else None

    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    fig.suptitle(title + "  — Espectrograma", fontsize=12, fontweight="bold")

    for i, (ax, label, col, color) in enumerate(zip(axes, _LABELS, _COLS, _COLORS.values())):
        sig = data[:, col].astype(float)
        f, t_spec, Sxx = scipy_spectrogram(sig, fs=sr, nperseg=NPERSEG, noverlap=NOVERLAP)
        Sxx_db = 10 * np.log10(Sxx + 1e-12)

        if t0 is not None:
            t_plot = [t0 + timedelta(seconds=float(tt)) for tt in t_spec]
            is_dt  = True
        else:
            t_plot = t_spec
            is_dt  = False

        im = ax.pcolormesh(t_plot, f, Sxx_db, shading="gouraud", cmap="inferno")
        ax.set_ylim(0, sr / 2)
        ax.set_ylabel(f"{label}\nFrecuencia (Hz)", fontsize=9)
        cbar = fig.colorbar(im, ax=ax, pad=0.01)
        cbar.set_label("dB re g²/Hz", fontsize=8)

        if i == 2:
            _apply_xaxis(ax, t_plot, is_dt)

    plt.tight_layout()
    _save(fig, save_path)
    plt.show()


# ──────────────────────────────────────────────────────────────────────────────
def plot_sta_lta(header, data, title, use_datetime, starttime, save_path=None):
    """
    Detección STA/LTA sobre la señal compuesta √(ax²+ay²+az²).
    Panel superior: señal compuesta en g.
    Panel inferior: función característica STA/LTA con umbrales ON/OFF.
    Las zonas de evento detectadas se resaltan en rojo transparente.
    """
    if use_datetime:
        xdata, is_dt = make_datetime_axis(data, starttime)
    else:
        xdata, is_dt = make_relative_axis(data), False

    composita = np.sqrt(data[:, 1]**2 + data[:, 2]**2 + data[:, 3]**2)
    cft       = classic_sta_lta(composita, STA_SAMPLES, LTA_SAMPLES)
    on_off    = trigger_onset(cft, TRIGGER_ON, TRIGGER_OFF)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    fig.suptitle(title + "  — STA/LTA", fontsize=12, fontweight="bold")

    # Panel superior — señal compuesta
    ax1.plot(xdata, composita, color="#455A64", linewidth=0.7)
    ax1.set_ylabel("√(ax²+ay²+az²) [g]", fontsize=9)
    ax1.grid(True, alpha=0.25)

    # Panel inferior — función característica
    ax2.plot(xdata, cft, color="#7B1FA2", linewidth=0.8, label="CFT")
    ax2.axhline(TRIGGER_ON,  color="red",    linewidth=1.0, linestyle="--", label=f"ON = {TRIGGER_ON}")
    ax2.axhline(TRIGGER_OFF, color="orange", linewidth=1.0, linestyle="--", label=f"OFF = {TRIGGER_OFF}")
    ax2.set_ylabel("STA / LTA", fontsize=9)
    ax2.legend(fontsize=8, loc="upper right")
    ax2.grid(True, alpha=0.25)

    # Sombrear zonas de evento en ambos paneles
    for on_idx, off_idx in on_off:
        x_on  = xdata[min(on_idx,  len(xdata) - 1)]
        x_off = xdata[min(off_idx, len(xdata) - 1)]
        for panel in (ax1, ax2):
            panel.axvspan(x_on, x_off, alpha=0.15, color="red")

    n_eventos = len(on_off)
    ax1.set_title(f"Eventos detectados: {n_eventos}", fontsize=9, pad=2)

    _apply_xaxis(ax2, xdata, is_dt)
    plt.tight_layout()
    _save(fig, save_path)
    plt.show()


print("Funciones de graficación definidas.")

---

# Carpeta `aceleraciones`

Grabaciones continuas del sensor (ruido ambiental, calibración, pruebas en campo, etc.).

## §11 — Aceleraciones · Series de tiempo en **g** · eje relativo (desde 0 s)

In [ ]:
for path in bins_acel:
    stem  = path.stem
    t0    = get_starttime(stem)
    hdr, data = parse_bin(path)

    save_path = DIR_PLOTS / f"{stem}_acel_g_relativo.png"
    plot_timeseries(
        hdr, data,
        title       = f"{stem}  —  Aceleración [g]  |  t relativo",
        use_datetime= False,
        starttime   = t0,
        save_path   = save_path,
    )

## §12 — Aceleraciones · Series de tiempo en **g** · eje fecha/hora real

In [ ]:
for path in bins_acel:
    stem  = path.stem
    t0    = get_starttime(stem)
    hdr, data = parse_bin(path)

    save_path = DIR_PLOTS / f"{stem}_acel_g_datetime.png"
    plot_timeseries(
        hdr, data,
        title       = f"{stem}  —  Aceleración [g]  |  hora real",
        use_datetime= True,
        starttime   = t0,
        save_path   = save_path,
    )

## §13 — Aceleraciones · Series de tiempo en **cm/s²** · eje relativo

In [ ]:
for path in bins_acel:
    stem  = path.stem
    t0    = get_starttime(stem)
    hdr, data = parse_bin(path)

    save_path = DIR_PLOTS / f"{stem}_acel_cms2_relativo.png"
    plot_timeseries_cms2(
        hdr, data,
        title       = f"{stem}  —  Aceleración  |  t relativo",
        use_datetime= False,
        starttime   = t0,
        save_path   = save_path,
    )

## §14 — Aceleraciones · Series de tiempo en **cm/s²** · eje fecha/hora real

In [ ]:
for path in bins_acel:
    stem  = path.stem
    t0    = get_starttime(stem)
    hdr, data = parse_bin(path)

    save_path = DIR_PLOTS / f"{stem}_acel_cms2_datetime.png"
    plot_timeseries_cms2(
        hdr, data,
        title       = f"{stem}  —  Aceleración  |  hora real",
        use_datetime= True,
        starttime   = t0,
        save_path   = save_path,
    )

## §15 — Aceleraciones · Espectrogramas (3 componentes)

In [ ]:
for path in bins_acel:
    stem  = path.stem
    t0    = get_starttime(stem)
    hdr, data = parse_bin(path)

    save_path = DIR_PLOTS / f"{stem}_acel_spectrogram.png"
    plot_spectrogram(
        hdr, data,
        title       = stem,
        use_datetime= True,
        starttime   = t0,
        save_path   = save_path,
    )

## §16 — Aceleraciones · Detección STA/LTA

In [ ]:
for path in bins_acel:
    stem  = path.stem
    t0    = get_starttime(stem)
    hdr, data = parse_bin(path)

    save_path = DIR_PLOTS / f"{stem}_acel_sta_lta.png"
    plot_sta_lta(
        hdr, data,
        title       = stem,
        use_datetime= True,
        starttime   = t0,
        save_path   = save_path,
    )

---

# Carpeta `eventos`

Archivos guardados por el nodo ESP32 después de un trigger STA/LTA. Deberían mostrar una señal claramente diferenciada del ruido de fondo.

## §17 — Eventos · Series de tiempo en **g** · eje relativo (desde 0 s)

In [ ]:
for path in bins_eventos:
    stem  = path.stem
    t0    = get_starttime(stem)
    hdr, data = parse_bin(path)

    save_path = DIR_PLOTS / f"{stem}_evento_g_relativo.png"
    plot_timeseries(
        hdr, data,
        title       = f"{stem}  —  Evento  [g]  |  t relativo",
        use_datetime= False,
        starttime   = t0,
        save_path   = save_path,
    )

## §18 — Eventos · Series de tiempo en **g** · eje fecha/hora real

In [ ]:
for path in bins_eventos:
    stem  = path.stem
    t0    = get_starttime(stem)
    hdr, data = parse_bin(path)

    save_path = DIR_PLOTS / f"{stem}_evento_g_datetime.png"
    plot_timeseries(
        hdr, data,
        title       = f"{stem}  —  Evento  [g]  |  hora real",
        use_datetime= True,
        starttime   = t0,
        save_path   = save_path,
    )

## §19 — Eventos · Series de tiempo en **cm/s²** · eje relativo

In [ ]:
for path in bins_eventos:
    stem  = path.stem
    t0    = get_starttime(stem)
    hdr, data = parse_bin(path)

    save_path = DIR_PLOTS / f"{stem}_evento_cms2_relativo.png"
    plot_timeseries_cms2(
        hdr, data,
        title       = f"{stem}  —  Evento  |  t relativo",
        use_datetime= False,
        starttime   = t0,
        save_path   = save_path,
    )

## §20 — Eventos · Series de tiempo en **cm/s²** · eje fecha/hora real

In [ ]:
for path in bins_eventos:
    stem  = path.stem
    t0    = get_starttime(stem)
    hdr, data = parse_bin(path)

    save_path = DIR_PLOTS / f"{stem}_evento_cms2_datetime.png"
    plot_timeseries_cms2(
        hdr, data,
        title       = f"{stem}  —  Evento  |  hora real",
        use_datetime= True,
        starttime   = t0,
        save_path   = save_path,
    )

## §21 — Eventos · Espectrogramas (3 componentes)

In [ ]:
for path in bins_eventos:
    stem  = path.stem
    t0    = get_starttime(stem)
    hdr, data = parse_bin(path)

    save_path = DIR_PLOTS / f"{stem}_evento_spectrogram.png"
    plot_spectrogram(
        hdr, data,
        title       = stem,
        use_datetime= True,
        starttime   = t0,
        save_path   = save_path,
    )

## §22 — Eventos · Detección STA/LTA

In [ ]:
for path in bins_eventos:
    stem  = path.stem
    t0    = get_starttime(stem)
    hdr, data = parse_bin(path)

    save_path = DIR_PLOTS / f"{stem}_evento_sta_lta.png"
    plot_sta_lta(
        hdr, data,
        title       = stem,
        use_datetime= True,
        starttime   = t0,
        save_path   = save_path,
    )

---

## §23 — Resumen estadístico

Tabla consolidada con métricas clave de todos los archivos procesados.

In [ ]:
print(f"\n{'Archivo':<42} {'Tipo':<8} {'Dur (s)':>8} {'PGA-EW (g)':>11} {'PGA-NS (g)':>11} {'PGA-VER (g)':>12} {'Trigger':>8} {'Lat':>9} {'Lon':>10}")
print("-" * 130)

for path, tag in all_bins:
    stem = path.stem
    try:
        hdr, data = parse_bin(path)
        dur    = len(data) / hdr["sample_rate"]
        pga_ew = np.max(np.abs(data[:, 1]))
        pga_ns = np.max(np.abs(data[:, 2]))
        pga_vr = np.max(np.abs(data[:, 3]))

        comp = np.sqrt(data[:, 1]**2 + data[:, 2]**2 + data[:, 3]**2)
        cft  = classic_sta_lta(comp, STA_SAMPLES, LTA_SAMPLES)
        on_off = trigger_onset(cft, TRIGGER_ON, TRIGGER_OFF)
        trigger_str = f"SI ({len(on_off)})" if on_off.size > 0 else "no"

        lat = f"{hdr['lat']:.4f}" if hdr['version'] >= 3 else "  —"
        lon = f"{hdr['lon']:.4f}" if hdr['version'] >= 3 else "  —"

        print(f"{path.name:<42} {tag:<8} {dur:>8.1f} {pga_ew:>11.4f} {pga_ns:>11.4f} {pga_vr:>12.4f} {trigger_str:>8} {lat:>9} {lon:>10}")
    except Exception as e:
        print(f"{path.name:<42} {'ERROR':<8} {str(e)}")

print("-" * 130)
print(f"Total archivos: {len(all_bins)}")